<h1 align="center">TÉCNICO EM CIÊNCIA DE DADOS</h1>

<h2 align="center">Roteiro de Atividade Prática</h2>

**Componente:** Análise Exploratória de Dados e Inteligência de Negócios  
**Unidade Curricular:** Projeto Análise Exploratória de Dados  
**Tema da Semana:** Projeto AED – Coleta e Preparação dos Dados  
**Semana 7 — Aula 4:** Validação da base preparada para análise

**Nome(s):** [preencher]  
**Turma:** [preencher]

**Objetivo:** Analisar a adequação da base preparada às perguntas e aos objetivos definidos para a investigação.


## Missão da atividade

Vocês atuam como assistentes de dados da equipe comercial de uma pequena rede brasileira de papelarias. A base é simulada e cada registro representa a operação de uma loja em um dia.

**Pergunta central:** Quais condições estão associadas ao desempenho comercial das lojas e qual aspecto merece prioridade para uma primeira ação de melhoria?

A S6 indicou a necessidade de aprofundar a investigação por **loja e faixa de movimento**. A redução da espera apareceu como um exemplo de prioridade fundamentada, não como conclusão obrigatória. Nesta semana, os dados foram selecionados, verificados e preparados.

**Missão:** Validar para quais usos a base preparada permite prosseguir, sob quais condições e quais pendências ainda precisam ser aprofundadas antes de ampliar as conclusões.

Trabalhem em duplas. Em cada etapa, um integrante executa as células e o outro confere as saídas e a justificativa. Alternem os papéis antes de registrar a decisão.

**Tempo total: 43 minutos.** As etapas duram 4, 8, 9, 10, 7 e 5 minutos.

A prioridade discutida na S6 não precisa ser confirmada. As novas evidências podem confirmá-la, qualificá-la ou levar à sua revisão. Associação não demonstra causalidade, e os resultados se restringem ao conjunto simulado.


## 1. Conheçam os critérios de validação — 4 minutos

Arquivos usados nesta atividade:

- `DADOS3EMC3B2S7A4_aluno.ipynb`
- `rede_papelarias_B2_integrado.csv`

Deixem o notebook e o CSV na mesma pasta. No Visual Studio Code, selecionem um kernel Python disponível. No Google Colab, façam o upload do CSV para a sessão. Executem as células na ordem e **não alterem o código**.

O notebook recompõe uma **base preparada de referência** usando apenas procedimentos já conhecidos: retira a cópia integral confirmada e recupera `cupons_por_100_entradas` e `margem_bruta_pct`. Nenhuma ausência é preenchida, nenhuma despesa é convertida e nenhum possível valor atípico é removido.

A validação usa três classificações:

- **Pode prosseguir:** os dados estão organizados e disponíveis para o uso definido.
- **Pode prosseguir com restrição:** o uso é possível, mas há uma condição ou limite que precisa acompanhar a interpretação.
- **Ainda depende de pendência:** falta resolver um problema necessário para aquele uso.

Durante as etapas 2 a 5, sustentem cada decisão com **evidência da saída + finalidade + limite**.


In [ ]:
from pathlib import Path
import pandas as pd
try:
    from IPython.display import display
except ImportError:
    def display(valor):
        print(valor)

nome_preferencial = "rede_papelarias_B2_integrado.csv"
arquivo_csv = Path(nome_preferencial)

if not arquivo_csv.is_file():
    candidatos = sorted(Path.cwd().glob("*rede_papelarias_B2_integrado*.csv"))
    if len(candidatos) != 1:
        raise FileNotFoundError(
            "Deixe o CSV integrado desta atividade na mesma pasta do notebook "
            "ou faça o upload para a sessão do Colab."
        )
    arquivo_csv = candidatos[0]

dados_originais = pd.read_csv(arquivo_csv)

# Base preparada de referência: recompõe somente procedimentos já conhecidos.
dados_preparados = dados_originais.drop_duplicates().copy()

if (dados_preparados["entradas"] <= 0).any() or (dados_preparados["receita_liquida"] <= 0).any():
    raise ValueError("Há denominadores não positivos; interrompa e avise o professor.")

dados_preparados["cupons_por_100_entradas"] = (
    100 * dados_preparados["cupons_emitidos"] / dados_preparados["entradas"]
)
dados_preparados["margem_bruta_pct"] = (
    100 * dados_preparados["lucro_bruto"] / dados_preparados["receita_liquida"]
)

faixa_inicio, faixa_fim = 150, 199
dados_faixa = dados_preparados[
    dados_preparados["entradas"].between(faixa_inicio, faixa_fim)
].copy()

print("Arquivo carregado:", arquivo_csv.name)
print("Registros originais:", len(dados_originais))
print("Registros na base preparada:", len(dados_preparados))
print("Colunas originais:", dados_originais.shape[1])
print("Colunas na base preparada:", dados_preparados.shape[1])
print("Faixa de movimento retomada do projeto: 150–199 entradas")


## 2. Validem as comparações por loja e faixa de movimento — 8 minutos

A S6 mostrou que resultados globais não descrevem automaticamente o que ocorre em cada loja. Por isso, o aprofundamento precisa considerar **loja** e uma **faixa comum de movimento**.

A faixa de **150–199 entradas** já foi utilizada no projeto porque apresentou a melhor cobertura mínima por loja entre as faixas comparadas. Nesta aula, não escolham outra faixa: verifiquem se o estado preparado da base continua sustentando esse uso.


In [ ]:
ordem_lojas = sorted(dados_preparados["loja"].dropna().unique())

cobertura = pd.DataFrame({
    "registros_totais": dados_preparados.groupby("loja").size().reindex(ordem_lojas),
    "registros_150_199": dados_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
    "ausentes_cupons_por_100_na_faixa": dados_faixa.groupby("loja")["cupons_por_100_entradas"].apply(lambda s: int(s.isna().sum())).reindex(ordem_lojas, fill_value=0),
    "ausentes_margem_bruta_na_faixa": dados_faixa.groupby("loja")["margem_bruta_pct"].apply(lambda s: int(s.isna().sum())).reindex(ordem_lojas, fill_value=0),
})

print("Cobertura por loja:")
display(cobertura)
print("Total de registros na faixa 150–199:", len(dados_faixa))
print("Menor número de registros entre as lojas nessa faixa:", int(cobertura["registros_150_199"].min()))
print("Lojas representadas na faixa:", int((cobertura["registros_150_199"] > 0).sum()), "de", len(ordem_lojas))


### Registro — Q1

**A base preparada permite aprofundar comparações por loja na faixa de 150–199 entradas?**

- **Classificação:** [Pode prosseguir / Pode prosseguir com restrição / Ainda depende de pendência]
- **Evidência 1:** [preencher]
- **Evidência 2:** [preencher]
- **Condição ou limite:** [preencher]

Não concluam que uma diferença observada dentro dessa faixa representa automaticamente todas as condições de operação das lojas.


## 3. Validem a investigação sobre espera e aproveitamento — 9 minutos

A S6 encontrou uma associação relevante entre espera e aproveitamento do movimento. Para aprofundá-la por loja e faixa de movimento, precisamos saber se há pares observados de `espera_media_min` e `cupons_por_100_entradas` em quantidade suficiente para continuar a investigação.

Nesta etapa, **não calculem correlação nem façam gráfico**. O objetivo é avaliar a adequação dos dados à análise futura.


In [ ]:
par_espera = ["espera_media_min", "cupons_por_100_entradas"]
pares_base = dados_preparados.dropna(subset=par_espera)
pares_faixa = dados_faixa.dropna(subset=par_espera)

cobertura_espera_faixa = pd.DataFrame({
    "registros_na_faixa": dados_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
    "pares_espera_aproveitamento": pares_faixa.groupby("loja").size().reindex(ordem_lojas, fill_value=0),
})
cobertura_espera_faixa["espera_ausente_na_faixa"] = (
    cobertura_espera_faixa["registros_na_faixa"] - cobertura_espera_faixa["pares_espera_aproveitamento"]
)

resumo_espera = pd.DataFrame({
    "situação": [
        "Base preparada",
        "Pares observados na base",
        "Faixa 150–199",
        "Pares observados na faixa",
    ],
    "registros": [
        len(dados_preparados),
        len(pares_base),
        len(dados_faixa),
        len(pares_faixa),
    ],
})

print("Cobertura da espera:")
display(resumo_espera)
print("Por loja, na faixa 150–199:")
display(cobertura_espera_faixa)
print("Esperas ausentes na base preparada:", int(dados_preparados["espera_media_min"].isna().sum()))
print("Esperas ausentes na faixa 150–199:", int(dados_faixa["espera_media_min"].isna().sum()))
print("Menor número de pares observados por loja na faixa:", int(cobertura_espera_faixa["pares_espera_aproveitamento"].min()))


### Registro — Q2

**Os dados permitem investigar espera × aproveitamento por loja e faixa de movimento?**

- **Classificação:** [Pode prosseguir / Pode prosseguir com restrição / Ainda depende de pendência]
- **Evidência:** [preencher]
- **Como as ausências limitam a abrangência:** [preencher]
- **O que deve ser preservado:** [preencher]

Lembrem-se: espera ausente significa **informação desconhecida**, não espera igual a zero.


## 4. Validem o aprofundamento do resultado comercial — 10 minutos

A base ampliada acrescentou informações de despesas. Agora precisamos distinguir três usos diferentes:

1. comparar **margem bruta**;
2. investigar um **resultado após despesas operacionais**;
3. decidir como tratar um **valor de receita muito distante dos demais**.

Executem a conferência abaixo. Não convertam despesas, não preencham ausências e não excluam o registro destacado.


In [ ]:
# Margem bruta: disponibilidade para uso analítico já conhecido.
resumo_margem = pd.DataFrame({
    "variavel": ["margem_bruta_pct"],
    "valores_disponiveis": [int(dados_preparados["margem_bruta_pct"].notna().sum())],
    "ausentes": [int(dados_preparados["margem_bruta_pct"].isna().sum())],
})
print("Margem bruta percentual:")
display(resumo_margem)

# Despesas: somente diagnóstico de disponibilidade e representação.
linhas_despesas = []
for coluna in ["despesas_com_pessoal", "outras_despesas_operacionais"]:
    textos = dados_preparados[coluna].dropna().astype(str)
    linhas_despesas.append({
        "coluna": coluna,
        "tipo_lido": str(dados_preparados[coluna].dtype),
        "ausentes": int(dados_preparados[coluna].isna().sum()),
        "valores_com_R$": int(textos.str.contains("R$", regex=False).sum()),
        "valores_com_virgula": int(textos.str.contains(",", regex=False).sum()),
    })
print("Pendências das despesas:")
display(pd.DataFrame(linhas_despesas))
print("Contagens de formato não devem ser somadas às ausências.")

print("Registro de receita destacado no diagnóstico:")
display(
    dados_preparados.loc[
        dados_preparados["id_registro"].eq("R03047"),
        ["id_registro", "data", "loja", "receita_liquida", "lucro_bruto", "margem_bruta_pct"],
    ].round(2)
)


### Registro — Q3

Para cada uso abaixo, registrem **se pode prosseguir agora** e justifiquem com a saída.

**a) Comparar margem bruta entre observações ou grupos**  
Classificação e justificativa: [preencher]

**b) Calcular um resultado após as despesas operacionais acrescentadas à base**  
Classificação e justificativa: [preencher]

**c) Excluir o registro R03047 apenas porque sua receita é muito diferente das demais**  
Classificação e justificativa: [preencher]

**Encaminhamento das pendências:**

- ausências e possível valor atípico → [preencher]
- problemas de representação/formato → [preencher]


## 5. Consolidem o diagnóstico de adequação — 7 minutos

Executem a célula de síntese das evidências e, em seguida, completem o Registro de validação.

A decisão deve variar conforme o **uso pretendido**. Evitem conclusões gerais como “a base está limpa” ou “a base está pronta para qualquer análise”.


In [ ]:
evidencias_validacao = pd.DataFrame([
    {
        "uso": "Comparações por loja e faixa 150–199",
        "evidencia_observada": f"{len(dados_faixa)} loja-dias; 8 lojas; mínimo de {int(cobertura['registros_150_199'].min())} registros por loja",
    },
    {
        "uso": "Espera × aproveitamento na faixa 150–199",
        "evidencia_observada": f"{len(pares_faixa)} pares observados; {int(dados_faixa['espera_media_min'].isna().sum())} esperas ausentes na faixa",
    },
    {
        "uso": "Margem bruta",
        "evidencia_observada": f"{int(dados_preparados['margem_bruta_pct'].notna().sum())} valores disponíveis de margem_bruta_pct",
    },
    {
        "uso": "Resultado após despesas operacionais",
        "evidencia_observada": "despesas com ausências e formatos mistos; cálculo ampliado ainda não preparado",
    },
])

display(evidencias_validacao)


### Registro de validação — Q4

Completem a matriz abaixo.

| Uso na investigação | Evidência de adequação | Decisão | Condição ou limite | Próxima etapa |
|---|---|---|---|---|
| Comparações por loja e faixa 150–199 | [preencher] | [preencher] | [preencher] | [preencher] |
| Espera × aproveitamento | [preencher] | [preencher] | [preencher] | [preencher] |
| Margem bruta | [preencher] | [preencher] | [preencher] | [preencher] |
| Resultado após despesas operacionais | [preencher] | [preencher] | [preencher] | [preencher] |

**Q4. Diagnóstico final:** Em uma frase, diferenciem o que já pode prosseguir do que ainda depende de pendência.  
**Resposta:** [preencher]

Fatores como organização da equipe, perfil do público, mix de produtos, eventos locais e características do atendimento podem continuar relevantes para explicar diferenças, mas não são “valores ausentes” nas colunas atuais. São informações ainda não registradas ou não demonstradas como disponíveis.


## 6. Revisem e comuniquem a validação — 5 minutos

**Revisão entre duplas — 2 min:** troquem o diagnóstico e questionem uma decisão de adequação.  
**Ajustes — 2 min:** melhorem a justificativa ou sustentem sua manutenção com evidência.  
**Síntese — 1 min:** respondam à equipe comercial em que condições a investigação pode continuar.

### Registro da revisão — Q5

**Observação recebida:** [preencher]

**Ajuste realizado ou justificativa para manter a decisão:** [preencher]

### Síntese da aprendizagem — Q6

A equipe comercial pergunta:

> **“Com a base preparada, quais partes do aprofundamento definido no projeto podem prosseguir, sob quais condições e o que ainda precisa ser resolvido?”**

**Resposta:** [preencher]

Não transformem a validação em recomendação final de intervenção. A prioridade discutida anteriormente pode ser confirmada, qualificada ou revista quando novas evidências forem produzidas.


## Critérios para avaliação da atividade

- Analisa a adequação da base em relação às perguntas e aos usos definidos para o aprofundamento da investigação.
- Sustenta decisões de adequação com evidências observáveis de cobertura, disponibilidade e preparação dos dados.
- Distingue usos que podem prosseguir, usos condicionados e usos que ainda dependem de pendências.
- Encaminha ausências e possíveis valores atípicos para aprofundamento na S8 e problemas de representação ou formato para a S9, sem antecipar seus tratamentos.
- Revê a própria decisão com outra dupla e preserva os limites da recomendação, sem afirmar causalidade ou uma prioridade comercial definitiva.

**Registro da atividade:** diagnóstico de adequação, revisão entre duplas e síntese final preenchidos no notebook e discutidos durante esta aula, sem exportação de dados para a próxima aula.

**Fonte dos dados:** base integrada do projeto SEDUC C3 B2, simulada para fins didáticos. O arquivo `rede_papelarias_B2_integrado.csv` é recomposto no notebook como base preparada de referência para validação.
